<a href="https://colab.research.google.com/github/AltaCedeno/ALTA/blob/main/notebook_para_Creaci%C3%B3n_mapa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Notebook Funcional: Generación de Polígonos UTM y Mapa Folium

#### 0. Instalación de librerías

In [ ]:
!pip install pandas geopandas shapely folium

#### 1. Importar librerías y subir CSV

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon
from IPython.display import display
from google.colab import files
import folium
import os

# Subir el archivo CSV
print('Por favor, selecciona tu archivo CSV:')
uploaded = files.upload()

# Asumiendo que solo se sube un archivo CSV
csv_filename = list(uploaded.keys())[0]
print(f"Archivo '{csv_filename}' subido exitosamente.")

# Leer el archivo CSV en un DataFrame de pandas
df = pd.read_csv(csv_filename)

display(df.head())
print(f"Columnas disponibles: {df.columns.tolist()}")

#### 2. Preparar el GeoDataFrame y crear puntos

In [ ]:
# Asegúrate de que los nombres de las columnas de coordenadas sean correctos.
# Ajusta 'longitud_col' y 'latitud_col' si tus columnas tienen nombres diferentes.
longitud_col = 'longitud' # Reemplaza si el nombre de tu columna es diferente
latitud_col = 'latitud'   # Reemplaza si el nombre de tu columna es diferente

# Crear puntos a partir de las coordenadas
df['geometry'] = df.apply(lambda row: Point(row[longitud_col], row[latitud_col]), axis=1)

# Crear un GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:32619") # CRS para UTM Zone 19N

display(gdf.head())

#### 3. Crear polígonos por 'ruta' e incluir `id_trampa` y calcular área

In [ ]:
# Asegúrate de que los nombres de las columnas 'ruta' y 'id_trampa' sean correctos.
# Ajusta 'ruta' y 'id_trampa' si tus columnas tienen nombres diferentes.
ruta = 'ruta'       # Reemplaza si el nombre de tu columna es diferente
id_trampa = 'id_trampa' # Reemplaza si el nombre de tu columna es diferente

# Verificar si las columnas existen
if ruta not in gdf.columns:
    raise ValueError(f"La columna '{ruta}' no se encontró en el DataFrame. Por favor, verifica el nombre de la columna.")
if id_trampa not in gdf.columns:
    raise ValueError(f"La columna '{id_trampa}' no se encontró en el DataFrame. Por favor, verifica el nombre de la columna.")

# Agrupar por 'ruta' y crear polígonos (casco convexo) y agregar id_trampa
polygons_gdf = gdf.groupby(ruta).agg({
    'geometry': lambda x: x.unary_union.convex_hull, # Crea el casco convexo de todos los puntos en el grupo
    id_trampa: lambda x: ', '.join(x.astype(str).unique()) # Concatena todos los id_trampa únicos en una cadena
}).reset_index()

# Asegurar que el resultado sea un GeoDataFrame con el CRS correcto
polygons_gdf = gpd.GeoDataFrame(polygons_gdf, geometry='geometry', crs=gdf.crs)

# Calcular el área de cada polígono en metros cuadrados (ya que el CRS es UTM)
polygons_gdf['area_sq_m'] = polygons_gdf.geometry.area

display(polygons_gdf.head())
print(f"Número de polígonos creados: {len(polygons_gdf)}")

#### 4. Guardar el archivo Shapefile

In [ ]:
# Reproyectar a WGS84 (EPSG:4326) para una mayor compatibilidad si es necesario para otros GIS
polygons_gdf_wgs84 = polygons_gdf.to_crs(epsg=4326)

# Nombre de la carpeta para el shapefile
output_folder = 'shapefile_rutas'
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Nombre del archivo shapefile
output_shapefile = os.path.join(output_folder, 'poligonos_rutas.shp')

# Guardar el GeoDataFrame como un shapefile
polygons_gdf_wgs84.to_file(output_shapefile, driver='ESRI Shapefile')

print(f"Shapefile guardado exitosamente en: {output_shapefile}")
print("Puedes descargar la carpeta 'shapefile_rutas' haciendo clic en el icono de la carpeta a la izquierda.")

#### 5. Visualizar los polígonos en Colab con Folium y guardar HTML

In [ ]:
import random

# Calcular el centroide de todos los polígonos usando el CRS proyectado original
# Esto evita la UserWarning al calcular el centroide en un CRS geográfico
polygons_gdf_projected = polygons_gdf.to_crs(polygons_gdf.crs)
center_lat_projected = polygons_gdf_projected.geometry.centroid.y.mean()
center_lon_projected = polygons_gdf_projected.geometry.centroid.x.mean()

# Reproyectar este punto central a WGS84 para Folium
center_point_wgs84 = gpd.GeoSeries([Point(center_lon_projected, center_lat_projected)], crs=polygons_gdf.crs).to_crs(epsg=4326)
center_lat = center_point_wgs84.y[0]
center_lon = center_point_wgs84.x[0]

m = folium.Map(location=[center_lat, center_lon], zoom_start=10)

# Generar colores únicos para cada ruta
num_routes = len(polygons_gdf_wgs84[ruta].unique())
colors = ['#%06x' % (random.randint(0, 0xFFFFFF)) for _ in range(num_routes)]
color_map = dict(zip(polygons_gdf_wgs84[ruta].unique(), colors))


for idx, row in polygons_gdf_wgs84.iterrows():
    # Asegurar que la geometría sea un polígono o multipolígono válido
    if row.geometry.geom_type == 'Polygon':
        geo_json_data = folium.GeoJson(
            row.geometry.__geo_interface__,
            style_function=lambda x, color=color_map[row[ruta]]: {
                'fillColor': color,
                'color': 'black',
                'weight': 1,
                'fillOpacity': 0.7
            }
        )
    elif row.geometry.geom_type == 'MultiPolygon':
        # Folium maneja bien MultiPolygons directamente
        geo_json_data = folium.GeoJson(
            row.geometry.__geo_interface__,
            style_function=lambda x, color=color_map[row[ruta]]: {
                'fillColor': color,
                'color': 'black',
                'weight': 1,
                'fillOpacity': 0.7
            }
        )
    else:
        # Manejar otros tipos de geometría si aparecen, o saltar
        continue

    # Formatear el área para el popup
    area_formatted = f"{row['area_sq_m']:,.2f} m²"

    # Crear el popup incluyendo el área
    popup_text = f"<b>Ruta:</b> {row[ruta]}<br><b>ID Trampa(s):</b> {row[id_trampa]}<br><b>Área:</b> {area_formatted}"
    popup = folium.Popup(popup_text, max_width=300)

    geo_json_data.add_child(popup)
    geo_json_data.add_to(m)

# Añadir leyenda
legend_html = '''
     <div style="position: fixed;
                 top: 50px; left: 50px; width: 150px; height: auto;
                 border:2px solid grey; z-index:9999; font-size:14px;
                 background-color:white; opacity:0.9;">
       &nbsp;<b>Leyenda de Rutas</b><br>
       '''
for route_name, color_code in color_map.items():
    legend_html += f'''
           &nbsp;<i style="background:{color_code}; border:1px solid grey;">&nbsp;&nbsp;&nbsp;&nbsp;</i>&nbsp; {route_name}<br>
           '''
legend_html += '''
     </div>
     '''
m.get_root().html.add_child(folium.Element(legend_html))

display(m)

map_filename = 'mapa_rutas.html'
m.save(map_filename)
print(f"Mapa HTML guardado como '{map_filename}'. Puedes descargarlo haciendo clic en el icono de la carpeta a la izquierda y arrastrando el archivo a tu escritorio.")

### 0. Instalación de librerías

Instalamos las librerías necesarias para el procesamiento geoespacial y la visualización.

In [1]:
!pip install pandas geopandas shapely folium

### 1. Subir el archivo CSV

Primero, necesitamos subir el archivo CSV que contiene las coordenadas. Por favor, ejecuta la siguiente celda y selecciona tu archivo CSV.

In [8]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon
from IPython.display import display
from google.colab import files
import folium
import os

# Subir el archivo CSV
uploaded = files.upload()

# Asumiendo que solo se sube un archivo CSV
csv_filename = list(uploaded.keys())[0]
print(f"Archivo '{csv_filename}' subido exitosamente.")

# Leer el archivo CSV en un DataFrame de pandas
df = pd.read_csv(csv_filename)

display(df.head())
print(f"Columnas disponibles: {df.columns.tolist()}")

KeyboardInterrupt: 

### 2. Preparar el GeoDataFrame y crear puntos

Vamos a convertir las columnas de latitud y longitud en objetos de geometría de `shapely` y crear un `GeoDataFrame`. Asumiremos que las columnas se llaman 'longitud' y 'latitud' y que el CRS es `EPSG:32619` (WGS 84 / UTM zone 19N) para la República Dominicana.

In [15]:
# Asegúrate de que los nombres de las columnas de coordenadas sean correctos.
# Ajusta 'longitud_col' y 'latitud_col' si tus columnas tienen nombres diferentes.
longitud_col = 'longitud' # Reemplaza si el nombre de tu columna es diferente
latitud_col = 'latitud'   # Reemplaza si el nombre de tu columna es diferente

# Crear puntos a partir de las coordenadas
df['geometry'] = df.apply(lambda row: Point(row[longitud_col], row[latitud_col]), axis=1)

# Crear un GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:32619") # CRS para UTM Zone 19N

display(gdf.head())

,id_trampa,cod_revisor,ruta,longitud,latitud,geometry
0,LA9825,12,1,517325,2029788,POINT (517325 2029788)
1,LA6252,12,1,515604,2032674,POINT (515604 2032674)
2,LA7802,12,1,516051,2032332,POINT (516051 2032332)
3,LA9508,12,1,516389,2031696,POINT (516389 2031696)
4,LA6285,12,1,516004,2031776,POINT (516004 2031776)


### 3. Crear polígonos por 'ruta' e incluir `id_trampa`

Agruparemos los puntos por la columna 'ruta' y crearemos un polígono (usando el casco convexo) para cada grupo. También agregaremos los IDs de trampa (`id_trampa`) a cada polígono.

In [16]:
# Asegúrate de que los nombres de las columnas 'ruta' y 'id_trampa' sean correctos.
# Ajusta 'ruta_col' y 'id_trampa_col' si tus columnas tienen nombres diferentes.
ruta = 'ruta'       # Reemplaza si el nombre de tu columna es diferente
id_trampa = 'id_trampa' # Reemplaza si el nombre de tu columna es diferente

# Verificar si las columnas existen
if ruta not in gdf.columns:
    raise ValueError(f"La columna '{ruta}' no se encontró en el DataFrame. Por favor, verifica el nombre de la columna.")
if id_trampa not in gdf.columns:
    raise ValueError(f"La columna '{id_trampa}' no se encontró en el DataFrame. Por favor, verifica el nombre de la columna.")

# Agrupar por 'ruta' y crear polígonos (casco convexo) y agregar id_trampa
polygons_gdf = gdf.groupby(ruta).agg({
    'geometry': lambda x: x.unary_union.convex_hull, # Crea el casco convexo de todos los puntos en el grupo
    id_trampa: lambda x: ', '.join(x.astype(str).unique()) # Concatena todos los id_trampa únicos en una cadena
}).reset_index()

# Asegurar que el resultado sea un GeoDataFrame con el CRS correcto
polygons_gdf = gpd.GeoDataFrame(polygons_gdf, geometry='geometry', crs=gdf.crs)

# Calcular el área de cada polígono en metros cuadrados (ya que el CRS es UTM)
polygons_gdf['area_sq_m'] = polygons_gdf.geometry.area

# Si se desean 5 polígonos y hay más, podrías necesitar una lógica adicional aquí
# Por ahora, se generarán tantos polígonos como rutas únicas existan.

display(polygons_gdf.head())
print(f"Número de polígonos creados: {len(polygons_gdf)}")

/tmp/ipykernel_1174/3327546942.py:14: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  'geometry': lambda x: x.unary_union.convex_hull, # Crea el casco convexo de todos los puntos en el grupo


,ruta,geometry,id_trampa,area_sq_m
0,1,"POLYGON ((519909 2026934, 517855 2028974, 5169...","LA9825, LA6252, LA7802, LA9508, LA6285, LA6458...",5625288.0
1,2,"POLYGON ((511607 2033978, 511409 2034679, 5118...","LA8527, LA9503, LA9509, LA3868, LA8526, LA3869...",14543991.5
2,3,"POLYGON ((543120 2032328, 542775 2032396, 5375...","LA1501, LA1816, LA1460, LA1817, LA1515, LA2431...",20331925.0
3,4,"POLYGON ((540807 2031420, 540497 2031537, 5221...","LA7271, LA8800, LA3394, LA3387, LA174, LA9632,...",151134500.0
4,5,"POLYGON ((526666 2048487, 523092 2048686, 5231...","LA728, LA4529, LA3303, LA9779, LA2557, LA1674,...",50073153.0


Número de polígonos creados: 5


### 4. Guardar el archivo Shapefile

Ahora, guardaremos los polígonos en un archivo `.shp` profesional. Para ello, primero es recomendable reproyectar los datos a WGS84 (`EPSG:4326`) si el shapefile va a ser utilizado en sistemas que esperan coordenadas geográficas.

In [17]:
# Reproyectar a WGS84 (EPSG:4326) para una mayor compatibilidad si es necesario para otros GIS
polygons_gdf_wgs84 = polygons_gdf.to_crs(epsg=4326)

# Nombre de la carpeta para el shapefile
output_folder = 'shapefile_rutas'
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Nombre del archivo shapefile
output_shapefile = os.path.join(output_folder, 'poligonos_rutas.shp')

# Guardar el GeoDataFrame como un shapefile
polygons_gdf_wgs84.to_file(output_shapefile, driver='ESRI Shapefile')

print(f"Shapefile guardado exitosamente en: {output_shapefile}")
print("Puedes descargar la carpeta 'shapefile_rutas' haciendo clic en el icono de la carpeta a la izquierda.")

Shapefile guardado exitosamente en: shapefile_rutas/poligonos_rutas.shp
Puedes descargar la carpeta 'shapefile_rutas' haciendo clic en el icono de la carpeta a la izquierda.


### 5. Visualizar los polígonos en Colab

Utilizaremos `folium` para crear un mapa interactivo y visualizar los polígonos coloreados por 'ruta'. Los `id_trampa` se incluirán en los popups de cada polígono.

In [20]:
import random

# Calcular el centroide de todos los polígonos usando el CRS proyectado original
# Esto evita la UserWarning al calcular el centroide en un CRS geográfico
polygons_gdf_projected = polygons_gdf.to_crs(polygons_gdf.crs)
center_lat_projected = polygons_gdf_projected.geometry.centroid.y.mean()
center_lon_projected = polygons_gdf_projected.geometry.centroid.x.mean()

# Reproyectar este punto central a WGS84 para Folium
center_point_wgs84 = gpd.GeoSeries([Point(center_lon_projected, center_lat_projected)], crs=polygons_gdf.crs).to_crs(epsg=4326)
center_lat = center_point_wgs84.y[0]
center_lon = center_point_wgs84.x[0]

m = folium.Map(location=[center_lat, center_lon], zoom_start=10)

# Generar colores únicos para cada ruta
num_routes = len(polygons_gdf_wgs84[ruta].unique())
colors = ['#%06x' % (random.randint(0, 0xFFFFFF)) for _ in range(num_routes)]
color_map = dict(zip(polygons_gdf_wgs84[ruta].unique(), colors))


for idx, row in polygons_gdf_wgs84.iterrows():
    # Asegurar que la geometría sea un polígono o multipolígono válido
    if row.geometry.geom_type == 'Polygon':
        geo_json_data = folium.GeoJson(
            row.geometry.__geo_interface__,
            style_function=lambda x, color=color_map[row[ruta]]: {
                'fillColor': color,
                'color': 'black',
                'weight': 1,
                'fillOpacity': 0.7
            }
        )
    elif row.geometry.geom_type == 'MultiPolygon':
        # Folium maneja bien MultiPolygons directamente
        geo_json_data = folium.GeoJson(
            row.geometry.__geo_interface__,
            style_function=lambda x, color=color_map[row[ruta]]: {
                'fillColor': color,
                'color': 'black',
                'weight': 1,
                'fillOpacity': 0.7
            }
        )
    else:
        # Manejar otros tipos de geometría si aparecen, o saltar
        continue

    # Formatear el área para el popup
    area_formatted = f"{row['area_sq_m']:,.2f} m²" # Formato con coma para miles y dos decimales (corregido el espacio)

    # Crear el popup incluyendo el área
    popup_text = f"<b>Ruta:</b> {row[ruta]}<br><b>ID Trampa(s):</b> {row[id_trampa]}<br><b>Área:</b> {area_formatted}"
    popup = folium.Popup(popup_text, max_width=300)

    geo_json_data.add_child(popup)
    geo_json_data.add_to(m)

# Añadir leyenda
legend_html = '''
     <div style="position: fixed;
                 top: 50px; left: 50px; width: 150px; height: auto;
                 border:2px solid grey; z-index:9999; font-size:14px;
                 background-color:white; opacity:0.9;">
       &nbsp;<b>Leyenda de Rutas</b><br>
       '''
for route_name, color_code in color_map.items():
    legend_html += f'''
           &nbsp;<i style="background:{color_code}; border:1px solid grey;">&nbsp;&nbsp;&nbsp;&nbsp;</i>&nbsp; {route_name}<br>
           '''
legend_html += '''
     </div>
     '''
m.get_root().html.add_child(folium.Element(legend_html))

display(m)

In [22]:
map_filename = 'mapa_rutas.html'
m.save(map_filename)
print(f"Mapa HTML guardado como '{map_filename}'. Puedes descargarlo haciendo clic en el icono de la carpeta a la izquierda y arrastrando el archivo a tu escritorio.")

Mapa HTML guardado como 'mapa_rutas.html'. Puedes descargarlo haciendo clic en el icono de la carpeta a la izquierda y arrastrando el archivo a tu escritorio.
